# KTÜN Chatbot — Model Karşılaştırması

Lokalde NVIDIA GPU yok; `gemma2:2b` CPU'da soru başına 12-62 saniye alıyor ve
2B üstü model denenemiyor. Colab'ın T4'ünde 7-9B modeller 4-bit ile çalışır.

**Amaç:** lokale hangi modelin kurulacağına karar vermek ve rapora tablo çıkarmak.

Kod bu notebook'un içinde gömülü geliyor — **başka dosya yüklemene gerek yok.**

## Çalıştırmadan önce

1. **Runtime → Change runtime type → T4 GPU** (zorunlu)
2. `google/gemma-*` modellerini de ölçmek istiyorsan: huggingface.co →
   Settings → Access Tokens'tan bir token al, `gemma-2-2b-it` ve `gemma-2-9b-it`
   sayfalarında lisansı kabul et, sonra Colab'ın sol panelindeki 🔑 **Secrets**
   bölümüne `HF_TOKEN` adıyla ekle ve "Notebook access"i aç.
   *Bunu atlarsan hiçbir şey kırılmaz — gemma'lar atlanır, Qwen modelleri ölçülür.*

Sonra hücreleri sırayla çalıştır. Son hücre Markdown tablo basar, doğrudan
README'ye yapıştırılabilir.


## 1. Kurulum

**Hiçbir dosya yüklemene gerek yok.** Projenin Colab'da gereken kısmı (`bot/ index/ common/ eval/ data/sample/`) sıkıştırılıp bu hücrenin içine gömülü; hücre onu `/content/ktun`'a açıyor ve bağımlılıkları kuruyor. Ne zip, ne GitHub, ne token.

> Klasörü zip'leyip yüklemek işe yaramıyordu: klasörün tamamı ~760 MB (`data/models` 470 MB embedding önbelleği + `.git` 290 MB), Colab'ın ihtiyacı olan kısım ise 0.5 MB — üstelik Colab'ın sol paneli klasör değil **dosya** kabul ediyor.

⚠️ Gömülü kod üretildiği andaki hâlidir. Projede değişiklik yaptıysan `python notebooks/uret_ipynb.py` ile notebook'u yenile.

In [ ]:
!pip -q install sentence-transformers rank-bm25 transformers accelerate bitsandbytes

import base64
import io
import os
import sys
import tarfile

KOK = "/content/ktun"

# bot/ index/ common/ eval/ data/sample/ — tar.gz + base64.
# Parantez sart: satirlar bitisik string literal olarak birlesiyor.
PAYLOAD = (
    "..."
    "AA=="
)

paket = tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz")
try:
    # Python 3.12+ filtresiz extractall icin uyari basiyor; eski surumler
    # 'filter' argumanini tanimiyor. Ikisinde de calissin.
    paket.extractall(KOK, filter="data")
except TypeError:
    paket.extractall(KOK)
paket.close()

os.chdir(KOK)
if KOK not in sys.path:
    sys.path.insert(0, KOK)    # 'bot', 'index', 'common' import edilebilsin
os.environ["KTUN_KOK"] = KOK   # sonraki hucreler bunu kullaniyor


# google/gemma-* HF'de KAPALI repo, token ister. Colab Secrets'ta HF_TOKEN
# varsa aliniyor; yoksa 4. hucre o modelleri atlayip Qwen'lerle devam ediyor.
def gizli(ad):
    try:
        from google.colab import userdata
        return userdata.get(ad)
    except Exception:
        return None


hf = gizli("HF_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf

ornek = "data/sample/pages.sample.jsonl"
print("Kok         :", KOK)
print("Klasorler   :", sorted(d for d in os.listdir() if os.path.isdir(d)))
print("Ornek veri  :", "VAR" if os.path.exists(ornek) else "YOK — indeks kurulamaz")
print("HF_TOKEN    :", "var" if hf else "YOK — google/gemma-* atlanacak")

## 2. İndeks
`data/raw/` gitignore'da; Colab repodaki örnek veriyle çalışır.

In [ ]:
import os
os.chdir(os.environ["KTUN_KOK"])
!python -m index.build_index --input data/sample/pages.sample.jsonl

## 3. Backend
Colab'da Ollama yok. `ColabBackend`, `bot/llm.py` ile **aynı** `generate()` arayüzünü sağlıyor — `bot/` tarafında tek satır değişmiyor.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class ColabBackend:
    """bot/llm.py ile aynı arayüz: generate(prompt, ..., sistem=None) -> str"""

    def __init__(self, model_adi: str):
        self.model = model_adi
        self.tokenizer = AutoTokenizer.from_pretrained(model_adi)
        self.llm = AutoModelForCausalLM.from_pretrained(
            model_adi,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
            ),
            device_map="auto",
        )

    def hazir_mi(self):
        return True, "hazir"

    def generate(self, prompt, sicaklik=0.2, maks_token=220, sistem=None):
        mesajlar = ([{"role": "system", "content": sistem}] if sistem else [])
        mesajlar.append({"role": "user", "content": prompt})

        # Gemma sistem rolünü desteklemiyor — kuralları kullanıcı mesajına katıyoruz.
        if "gemma" in self.model.lower() and sistem:
            mesajlar = [{"role": "user", "content": f"{sistem}\n\n{prompt}"}]

        # transformers 4.4x ile apply_chat_template artik duz tensor degil
        # BatchEncoding donduruyor; .shape yok, AttributeError aliniyordu.
        # return_dict=True ile sozluk isteniyor, generate'e **ile aciliyor.
        girdi = self.tokenizer.apply_chat_template(
            mesajlar, add_generation_prompt=True, return_tensors="pt", return_dict=True
        )
        if torch.is_tensor(girdi):          # eski surumler: duz tensor
            girdi = {"input_ids": girdi}
        girdi = {ad: t.to(self.llm.device) for ad, t in girdi.items()}
        istem_uzunlugu = girdi["input_ids"].shape[-1]

        with torch.no_grad():
            cikti = self.llm.generate(
                **girdi,
                max_new_tokens=maks_token,
                temperature=max(sicaklik, 0.01),
                do_sample=sicaklik > 0,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        # Sadece uretilen kisim: istem cikti icinde tekrar geliyor.
        return self.tokenizer.decode(
            cikti[0][istem_uzunlugu:], skip_special_tokens=True
        ).strip()

## 4. Karşılaştırma
Her model aynı sorularla ölçülür. Model değişimi arasında GPU belleği boşaltılır, yoksa ikinci model sığmaz.

In [ ]:
import os, sys, time, gc, torch

# Guvenlik agi: hucreler sirasiz calistirilirsa veya oturum yeniden baglanirsa
# calisma dizini /content'e doner ve "No module named 'bot'" hatasi alinir.
KOK = os.environ.get("KTUN_KOK")
if not KOK:
    raise SystemExit("Once 1. hucreyi calistir (proje kokunu o buluyor).")
os.chdir(KOK)
if KOK not in sys.path:
    sys.path.insert(0, KOK)

from bot.answer import Chatbot
from eval.answer_test import SORULAR, KAPSAM_DISI

# Hepsi Türkçe destekli, Colab T4'e 4-bit ile sığan modeller.
ADAYLAR = [
    "google/gemma-2-2b-it",          # lokalde kazanan, referans
    "Qwen/Qwen2.5-1.5B-Instruct",    # lokalde hızlı ama isabetsiz
    "Qwen/Qwen2.5-7B-Instruct",      # lokalde çalışmaz, tavanı görmek için
    "google/gemma-2-9b-it",          # en büyük aday
]

# google/gemma-* HF'de kapali repo (lisans onayi + token ister). Token yoksa
# indirme 4 dakika sonra 401 ile duserdi; bastan eleniyor ki neden belli olsun.
if not os.environ.get("HF_TOKEN"):
    atlanan = [m for m in ADAYLAR if m.startswith("google/")]
    ADAYLAR = [m for m in ADAYLAR if m not in atlanan]
    if atlanan:
        print("HF_TOKEN yok, atlanan modeller:", ", ".join(atlanan))

sonuclar = []
for model_adi in ADAYLAR:
    print(f"\n{'='*70}\n{model_adi}\n{'='*70}")
    try:
        bot = Chatbot("dummy")            # indeks/retriever kurulsun
        bot.backend = ColabBackend(model_adi)   # backend'i degistir
    except Exception as e:
        print(f"  YUKLENEMEDI: {e}")
        continue

    isabet, sure_toplam = 0, 0.0
    for soru, beklenen, tip in SORULAR:
        bot.gecmisi_temizle()
        t = time.time()
        cevap = bot.sor(soru)
        sure_toplam += time.time() - t
        ok = beklenen.casefold() in cevap.metin.casefold()
        isabet += ok
        print(f"  {'OK  ' if ok else 'HATA'} {soru[:42]:44} {cevap.metin[:70]}")

    # Kapsam disi sorular LLM'e hic gitmiyor (kapi retriever'da), yine de dogrula.
    kapsam_ok = sum(bot.sor(s).kapsam_disi for s in KAPSAM_DISI)

    sonuclar.append({
        "model": model_adi,
        "dogruluk": f"{isabet}/{len(SORULAR)}",
        "yuzde": round(100 * isabet / len(SORULAR)),
        "kapsam": f"{kapsam_ok}/{len(KAPSAM_DISI)}",
        "sn_soru": round(sure_toplam / len(SORULAR), 1),
    })

    del bot
    gc.collect()
    torch.cuda.empty_cache()

## 5. Tablo
Çıktı doğrudan README'ye yapıştırılabilir.

In [ ]:
import pandas as pd
tablo = pd.DataFrame(sonuclar)
print(tablo.to_markdown(index=False))   # dogrudan README'ye yapistirilabilir
tablo.to_csv("model_karsilastirma.csv", index=False)